
# Test Core02 Notebook

This notebook tests the core_02 data extraction functionality from the pat2vec pipeline.

## Data Source
- **Source Index**: Multiple indices 
- **Content Type**: Core patient data features
- **Key Fields**: `client_idcode`, various clinical features

## Test Flow
1. Start Elasticsearch container with dummy data
2. Configure pat2vec to extract core_02 features using database backend
3. Process patients using the `pat_maker` pipeline
4. Retrieve all extracted features from SQLite database
5. Verify the merged output is non-empty


In [ ]:
import os
import random
import shutil
import sys

import numpy as np

In [ ]:
random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
sys.path.insert(0, pat2vec_dir)

print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
TEMP_BASE_DIR = "/tmp/core02_test_project"

try:
    shutil.rmtree(TEMP_BASE_DIR, ignore_errors=True)
except Exception as e:
    msg = f"Failed to clean up directory: {e}."
    raise RuntimeError(msg)

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container...")
if not es_container.start():
    msg = "Failed to start Elasticsearch."
    raise RuntimeError(msg)

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials.py"
creds_content = f"""\nusername = '{username}\'\npassword = '{password}\'\napi_key = None\nhosts = ["{host}"]\n"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created {creds_filename}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

grandparent_dir = "/workspaces/pat2vec"
schema_path = os.path.join(grandparent_dir, "test_files", "elastic_schemas.json")

config_populate = config_class(
    proj_name=TEMP_BASE_DIR,
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print(f"Population complete. Generated {len(patient_ids)} dummy patients.")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
import time

time.sleep(2)
print("Indices refreshed.")

In [ ]:
DB_FILENAME = "temp_core02_db.sqlite"
DB_PATH = os.path.join(TEMP_BASE_DIR, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    msg = f"Failed to remove database: {e}."
    raise RuntimeError(msg)

db_connection_string = "sqlite:///" + DB_PATH
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.main_pat2vec import main

try:
    config_obj = config_class(
        proj_name=TEMP_BASE_DIR,
        credentials_path=creds_filename,
        current_path_dir="",
        main_options={"core_02": True},
        batch_mode=True,
        verbosity=0,
        random_seed_val=random_seed_value,
        testing=True,
        testing_elastic=True,
        dummy_medcat_model=True,
        use_controls=False,
        medcat=False,
        start_time=None,
        patient_id_column_name="client_idcode",
        annot_filter_options={},
        shuffle_pat_list=False,
        storage_backend="database",
        check_patient_existence=False,
        db_connection_string=db_connection_string,
        treatment_doc_filename="test_files/treatment_docs.csv",
    )
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except Exception as e:
    msg = f"Failed to initialize: {e}"
    raise RuntimeError(msg)

In [ ]:
print("\n=== PROCESSING PATIENTS WITH pat_maker ===")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

try:
    print(f"Processing patient 0: {pat2vec_obj.all_patient_list[0]}")
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = (
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    )
    raise RuntimeError(
        msg,
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = (
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )
    raise RuntimeError(
        msg,
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
def merge_core02_data(config_obj):
    """Merge all core02 data and raise ValueError if empty.

    Args:
        config_obj: Configuration object with database connection info

    Returns:
        pd.DataFrame: Merged core02 feature data

    Raises:
        ValueError: If no core02 data was extracted or returned

    """
    all_features = get_all_features(config_obj)

    if len(all_features) == 0:
        msg = "MERGE FAILED: Merged core02 file is empty. No core02 data was returned."
        raise ValueError(
            msg,
        )

    return all_features

In [ ]:
try:
    merged_core02 = merge_core02_data(config_obj)
    print(f"Merged core02 data: {merged_core02.shape[0]} rows")
except ValueError as e:
    if "empty" in str(e).lower():
        msg = f"Merge function failed to process core02 data: {e}"
        raise RuntimeError(msg) from e
    raise

In [ ]:
import pandas as pd

from pat2vec.util.post_processing_build_methods import merge_core_02_csv

print("\n=== DEMONSTRATING FEATURE MERGE FUNCTIONALITY ===")
merged_path = merge_core_02_csv(
    pat2vec_obj.all_patient_list, config_obj, overwrite=True
)
merged_data = pd.read_csv(merged_path)
print(f"Merged core02 data saved to: {merged_path}")
print(f"Shape: {merged_data.shape}")
if merged_data.empty:
    raise RuntimeError(
        "FATAL ERROR: Merged core02 dataframe is empty. "
        "This indicates the pat2vec pipeline did not save data to database "
        "or no core02 features were found for the patients."
    )
print(f"\nColumns: {list(merged_data.columns)}")
print("\nData preview:")
print(merged_data.head())

In [ ]:
print("\n=== VERIFICATION CHECKS ===")

assert len(all_features) > 0, "Features DataFrame should have data"
print(f"✓ Features DataFrame has {len(all_features)} rows")

if "client_idcode" in all_features.columns:
    print("✓ client_idcode column exists in features")
else:
    msg = "client_idcode column missing from features"
    raise AssertionError(msg)

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    msg = f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(TEMP_BASE_DIR):
        shutil.rmtree(TEMP_BASE_DIR, ignore_errors=False)
        print(f"Removed project directory: {TEMP_BASE_DIR}")
except Exception as e:
    msg = f"Failed to remove '{TEMP_BASE_DIR}' directory: {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    msg = (
        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "
        "Critical error - cleanup incomplete."
    )
    raise RuntimeError(
        msg,
    ) from e

In [ ]:
assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(TEMP_BASE_DIR), "Project directory still exists!"
assert not os.path.exists(
    creds_filename,
), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")